# Tutorial 7: Train NicheTrans on 10x Xenium data

In [1]:
import os, time, datetime, warnings

import pandas as pd
import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_breast_cancer import Breast_cancer

from utils.utils import *
from utils.utils_training_breast_cancer import train, test
from utils.utils_dataloader import *
from prior_AddOn.gene_prior_ablation import randomize_static_gene_prior
from prior_AddOn.gene_embedding_loader import load_static_gene_prior
from prior_AddOn.gene_prior_filter import filter_dataset_by_gene_prior

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_breast_cancer.py
args = args

fixed_training_seed = args.seed
use_random_prior_ablation = False
random_prior_seeds = [0, 1, 2, 3, 4]

set_seed(fixed_training_seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))


Args:Namespace(noise_rate=0.2, dropout_rate=0.2, workers=4, adata_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nc_10x_breast_cancer/HBC_rep1_cell_nucleus_3channel_strength_mean.h5ad', coordinate_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nc_10x_breast_cancer/cells.csv.gz', ct_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nc_10x_breast_cancer/Cell_Barcode_Type_Matrices.xlsx', max_epoch=40, stepsize=20, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataset
dataset = Breast_cancer(adata_path=args.adata_path, coordinate_path=args.coordinate_path, ct_path=args.ct_path)

# load and apply static gene priors before creating dataloaders/model
prior_model = "scgpt"
priors = load_static_gene_prior(
    source_panel=dataset.source_panel,
    models=(prior_model,),
    species="human",
    root="prior_AddOn/gene_embeddings",
)
dataset, priors, filter_info = filter_dataset_by_gene_prior(
    dataset=dataset,
    priors=priors,
    prior_model=prior_model,
)
print(
    f"Gene prior filtering ({prior_model}): kept "
    f"{len(filter_info['filtered_source_panel'])}/"
    f"{len(filter_info['original_source_panel'])} genes; "
    f"removed {len(filter_info['removed_genes'])}."
)

source_dimension, target_dimension = dataset.rna_length, dataset.protein_length


------Calculating spatial graph...
The graph contains 1185564 edges, 98797 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 827796 edges, 68983 cells.
12.0000 neighbors per cell on average.
=> AD Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  98797 spots, 98659 positive CD20, 84043 positive HER2 
  test     |  68983 spots, 67600 positive CD20, 36904 positive HER2 
  ------------------------------
Gene prior filtering (scgpt): kept 311/313 genes; removed 2.


### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.MSELoss()


def build_model(run_priors):
    model = NicheTrans(
        source_length=source_dimension,
        target_length=target_dimension,
        noise_rate=args.noise_rate,
        dropout_rate=args.dropout_rate,
        priors=run_priors,
        prior_model=prior_model,
        prior_pooling_mode="qkv",
        normalize_prior_embedding=True,
    )
    return nn.DataParallel(model).cuda()


def build_optimizer(model):
    if args.optimizer == 'adam':
        return torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    if args.optimizer == 'SGD':
        return torch.optim.SGD(model.parameters(), lr=args.lr)
    raise ValueError(f"unexpected optimizer: {args.optimizer}")


def build_scheduler(optimizer):
    if args.stepsize > 0:
        return lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)
    return None


### Model training and testing

In [5]:
def run_training_experiment(run_priors, random_prior_seed=None):
    global model
    set_seed(fixed_training_seed)
    run_label = "static" if random_prior_seed is None else f"random_prior_seed_{random_prior_seed}"
    print(f"==> Starting {run_label}")

    trainloader, testloader = breast_cancer_dataloader(args, dataset)
    model = build_model(run_priors)
    optimizer = build_optimizer(model)
    scheduler = build_scheduler(optimizer)

    start_time = time.time()
    for epoch in range(args.max_epoch):
        print("==> Epoch {}/{}".format(epoch + 1, args.max_epoch))
        train(model, criterion, optimizer, trainloader)
        if scheduler is not None:
            scheduler.step()

    pearson_mean = test(model, testloader)
    checkpoint_path = (
        "NicheTrans_breast_cancer_last_vecnorm.pth"
        if random_prior_seed is None
        else f"NicheTrans_breast_cancer_random_prior_seed_{random_prior_seed}.pth"
    )
    torch.save(model.state_dict(), checkpoint_path)

    elapsed_seconds = round(time.time() - start_time)
    elapsed = str(datetime.timedelta(seconds=elapsed_seconds))
    print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

    return {
        "run": run_label,
        "random_prior_seed": random_prior_seed,
        "is_random_prior": random_prior_seed is not None,
        "pearson_mean": pearson_mean,
        "checkpoint": checkpoint_path,
        "elapsed_seconds": elapsed_seconds,
    }


experiment_runs = []
if use_random_prior_ablation:
    for random_prior_seed in random_prior_seeds:
        experiment_runs.append(
            (
                random_prior_seed,
                randomize_static_gene_prior(
                    priors=priors,
                    prior_model=prior_model,
                    seed=random_prior_seed,
                ),
            )
        )
else:
    experiment_runs.append((None, priors))

ablation_results = [
    run_training_experiment(run_priors, random_prior_seed=random_prior_seed)
    for random_prior_seed, run_priors in experiment_runs
]
results_df = pd.DataFrame(ablation_results)
metric_columns = ["pearson_mean"]
summary_df = results_df[metric_columns].agg(["mean", "std"]).T
summary_df["n_seeds"] = len(results_df)

display(results_df)
display(summary_df)


==> Epoch 1/40
Batch 3087/3087	 Loss 0.144300 (0.312058)
==> Epoch 2/40
Batch 3087/3087	 Loss 0.195039 (0.226911)
==> Epoch 3/40
Batch 3087/3087	 Loss 0.156631 (0.209457)
==> Epoch 4/40
Batch 3087/3087	 Loss 0.172250 (0.195470)
==> Epoch 5/40
Batch 3087/3087	 Loss 0.174852 (0.184462)
==> Epoch 6/40
Batch 3087/3087	 Loss 0.343933 (0.176639)
==> Epoch 7/40
Batch 3087/3087	 Loss 0.170167 (0.172880)
==> Epoch 8/40
Batch 3087/3087	 Loss 0.195077 (0.168393)
==> Epoch 9/40
Batch 3087/3087	 Loss 0.269808 (0.166764)
==> Epoch 10/40
Batch 3087/3087	 Loss 0.169125 (0.164650)
==> Epoch 11/40
Batch 3087/3087	 Loss 0.122279 (0.160640)
==> Epoch 12/40
Batch 3087/3087	 Loss 0.178969 (0.160980)
==> Epoch 13/40
Batch 3087/3087	 Loss 0.191788 (0.157491)
==> Epoch 14/40
Batch 3087/3087	 Loss 0.109840 (0.157471)
==> Epoch 15/40
Batch 3087/3087	 Loss 0.120671 (0.155301)
==> Epoch 16/40
Batch 3087/3087	 Loss 0.186371 (0.155048)
==> Epoch 17/40
Batch 3087/3087	 Loss 0.141086 (0.153765)
==> Epoch 18/40
Batch 3